# PlantAdvisor — baseline

Запуск в Google Colab на GPU T4. Выполняйте ячейки сверху вниз.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/Batyaoftheyear/PlantAdvisor"
REPO_REF = "main"
PROJECT = Path("/content/PlantAdvisor")
RUN_SANITY = True
RUN_FULL_EVAL = False
DOWNLOAD_RESULTS = False


## 1. Репозиторий


In [ ]:
if not PROJECT.exists():
    subprocess.run(["git", "clone", "--branch", REPO_REF, REPO_URL, str(PROJECT)], check=True)
if not (PROJECT / "pyproject.toml").is_file():
    raise RuntimeError("В рабочей папке отсутствует pyproject.toml")
os.chdir(PROJECT)
if (PROJECT / ".git").exists():
    subprocess.run(["git", "rev-parse", "HEAD"], check=True)
print("Рабочая папка:", PROJECT)


## 2. Зависимости


In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("Выберите GPU T4 в настройках Colab")
print("GPU:", torch.cuda.get_device_name(0), "torch:", torch.__version__, "CUDA:", torch.version.cuda)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements-gpu.txt"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--no-build-isolation", "-e", "."], check=True)


## 3. Проверка данных и кода


In [ ]:
subprocess.run([sys.executable, "-m", "plant_advisor", "audit"], check=True)
subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", "tests", "-v"], check=True)


## 4. Пробный запуск

Два вопроса. Ответы сохраняются в `outputs/sanity/`.


In [ ]:
if RUN_SANITY:
    subprocess.run([sys.executable, "-m", "plant_advisor", "run"], check=True)
else:
    print("Пробный запуск отключён")


## 5. Полный тест

После проверки пробных ответов задайте `RUN_FULL_EVAL=True`. Результаты: `outputs/runs/`.


In [ ]:
if RUN_FULL_EVAL:
    subprocess.run([sys.executable, "-m", "plant_advisor", "run", "--full", "--plot"], check=True)
else:
    print("Полный тест отключён: RUN_FULL_EVAL=False")


## 6. Скачать результаты

Задайте `DOWNLOAD_RESULTS=True`, чтобы скачать ZIP до завершения сессии Colab.


In [ ]:
if DOWNLOAD_RESULTS:
    import shutil
    from google.colab import files

    output_dir = PROJECT / "outputs"
    if not output_dir.is_dir():
        raise RuntimeError("Нет результатов для скачивания")
    archive = shutil.make_archive("/content/plant_advisor_results", "zip", root_dir=output_dir)
    files.download(archive)
else:
    print("Скачивание отключено: DOWNLOAD_RESULTS=False")
